# CodeMaya — Results & Figures
Reproduces the paper's figures/tables from `results/*.json`.

Set `CODEMAYA_RESULTS` to point at a results dir (default `outputs/results`). Run the pipeline (or `scripts/smoke.sh`) first to populate it.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RESULTS = Path(os.environ.get('CODEMAYA_RESULTS', 'outputs/results'))
def load(name):
    p = RESULTS / f'{name}.json'
    return json.load(open(p)) if p.exists() else None

print('results dir:', RESULTS.resolve())
print('available:', sorted(p.stem for p in RESULTS.glob('*.json')))

## Dataset — prompt-length distribution (Fig. 3)

In [ ]:
stats = load('dataset_stats')
if stats:
    h = stats['prompt_len']['histogram']
    plt.figure(figsize=(6,3))
    plt.bar([str(b) for b in h['bins']], h['counts'], color='#f0a15e')
    plt.title('Prompt length distribution (Fig. 3)'); plt.xlabel('token bin'); plt.ylabel('count')
    plt.tight_layout(); plt.show()
else:
    print('no dataset_stats.json')

## Top functions (Table 2)

In [ ]:
if stats and stats.get('top_functions'):
    tf = stats['top_functions'][:10]
    names=[t[0] for t in tf]; counts=[t[1] for t in tf]
    plt.figure(figsize=(6,3)); plt.barh(names[::-1], counts[::-1], color='#6ea8d8')
    plt.title('Top functions (Table 2)'); plt.tight_layout(); plt.show()

## Explainability split (Table 3)

In [ ]:
if stats and stats.get('explainability'):
    e = stats['explainability']
    plt.figure(figsize=(3,3))
    plt.pie([e['explained'], e['code_only']], labels=['explained','code-only'], autopct='%1.0f%%')
    plt.title('Explainability (Table 3)'); plt.show()

## Syntax validity per model (Table 1)

In [ ]:
sv = load('syntax_validity')
if sv:
    models=list(sv); vals=[sv[m]['valid_pct'] for m in models]
    plt.figure(figsize=(5,3)); plt.bar(models, vals, color='#7bc47f'); plt.ylim(0,100)
    plt.ylabel('% valid'); plt.title('Syntax validity (Table 1)')
    for i,v in enumerate(vals): plt.text(i, v+1, f'{v:.0f}%', ha='center')
    plt.tight_layout(); plt.show()
else:
    print('no syntax_validity.json')

## Semantic (CLIP) & visual (DINOv2) similarity

In [ ]:
sm = load('semantic_visual')
if sm:
    models=list(sm); clip=[sm[m]['clip_semantic'] for m in models]; dino=[sm[m]['dino_visual'] for m in models]
    x=np.arange(len(models)); w=0.35
    plt.figure(figsize=(5,3))
    plt.bar(x-w/2, clip, w, label='CLIP semantic'); plt.bar(x+w/2, dino, w, label='DINOv2 visual')
    plt.xticks(x, models); plt.legend(); plt.title('Semantic & visual similarity'); plt.tight_layout(); plt.show()
else:
    print('no semantic_visual.json (run eval-semantic)')

## Geometry metrics (Chamfer / Hausdorff / volume / area)

In [ ]:
geo = load('geometry')
if geo:
    models=list(geo); metrics=['chamfer','hausdorff','volume_ratio','area_ratio']; x=np.arange(len(metrics))
    plt.figure(figsize=(6,3))
    for i,m in enumerate(models):
        plt.bar(x+i*0.25, [geo[m][k] for k in metrics], 0.25, label=m)
    plt.xticks(x+0.12, metrics); plt.legend(); plt.title('Geometry metrics'); plt.tight_layout(); plt.show()
else:
    print('no geometry.json (run eval-geometry on real generations)')

## Siamese-ViT pairwise similarity (Table 5)

In [ ]:
si = load('siamese')
if si:
    pairs=list(si); vals=[si[p] for p in pairs]
    plt.figure(figsize=(5,3)); plt.bar(pairs, vals, color='#c98bd0'); plt.ylabel('cosine')
    plt.title('Siamese similarity (Table 5)'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()
else:
    print('no siamese.json')